# SLIS — Step 1: Data Generation
Generates 500 synthetic students with realistic correlated data across 4 CSVs:
- `students.csv` — demographics, GPA, archetype
- `attendance.csv` — monthly attendance (4 months)
- `scores.csv` — per-subject IT1/IT2/Final scores
- `activity.csv` — LMS engagement metrics

In [ ]:
import sys
from pathlib import Path

# Make sure we're running from the project root
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

np.random.seed(42)
N = 500
DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)
print('Data dir:', DATA_DIR)

## 1. Latent Variables & Archetypes

In [ ]:
MAJORS   = ['Computer Science', 'Data Science', 'Electronics', 'Mechanical', 'Civil', 'Mathematics']
GENDERS  = ['Male', 'Female', 'Non-binary']
SUBJECTS = ['Mathematics', 'Physics', 'Programming', 'Data Structures', 'Statistics']

FIRST_NAMES = [
    'Aarav','Vivaan','Aditya','Vihaan','Arjun','Sai','Reyansh','Ayaan','Krishna','Ishaan',
    'Priya','Ananya','Divya','Shreya','Pooja','Meera','Kavya','Lakshmi','Nandini','Riya',
    'Rohan','Kiran','Neha','Aryan','Tanvi','Rahul','Sneha','Vijay','Anjali','Suresh',
    'Omar','Liam','Noah','Emma','Olivia','Ava','Sophia','Isabella','Mia','Charlotte',
    'Amara','Zara','Fatima','Hassan','Yusuf','Leila','Tariq','Noor','Amir','Yasmin',
]
LAST_NAMES = [
    'Sharma','Patel','Singh','Kumar','Gupta','Joshi','Verma','Reddy','Nair','Rao',
    'Khan','Ali','Ahmed','Hassan','Shah','Malik','Chaudhary','Ansari','Siddiqui','Mirza',
    'Smith','Johnson','Williams','Brown','Jones','Davis','Miller','Wilson','Moore','Taylor',
]

def gen_name():
    return f"{np.random.choice(FIRST_NAMES)} {np.random.choice(LAST_NAMES)}"

latent_ability   = np.random.beta(2, 1.5, N)
latent_effort    = np.random.beta(1.8, 1.8, N)
latent_momentum  = np.random.normal(0, 0.18, N)
latent_anxiety   = np.random.beta(1.5, 3, N)
latent_social    = np.random.beta(1.5, 2, N)
latent_discipline = 0.5 * latent_effort + 0.2 * np.random.rand(N) + 0.3 * np.random.rand(N)

ARCHETYPES = np.random.choice(
    [0, 1, 2, 3, 4, 5, 6], N,
    p=[0.33, 0.13, 0.19, 0.13, 0.08, 0.08, 0.06]
)
arch_map = {0:'Consistent', 1:'Late Bloomer', 2:'Early Bird',
            3:'Struggle', 4:'Comeback', 5:'Exam Ace', 6:'Final Burnout'}
print('Archetypes:', {arch_map[k]: int((ARCHETYPES==k).sum()) for k in range(7)})

## 2. Generate students.csv

In [ ]:
student_ids = [f'STU{str(i+1).zfill(4)}' for i in range(N)]
names       = [gen_name() for _ in range(N)]
ages        = np.random.randint(15, 23, N)
genders     = np.random.choice(GENDERS, N, p=[0.48, 0.48, 0.04])
majors      = np.random.choice(MAJORS, N)
enroll_year = np.random.choice([2020,2021,2022,2023,2024], N, p=[0.1,0.15,0.25,0.3,0.2])
gpa_start   = np.clip(latent_ability * 7 + np.random.normal(1.5, 0.8, N), 1.0, 10.0).round(1)

students = pd.DataFrame({
    'student_id': student_ids, 'name': names, 'age': ages,
    'gender': genders, 'major': majors, 'enrollment_year': enroll_year,
    'gpa_start': gpa_start, 'archetype': ARCHETYPES,
})
students.to_csv(DATA_DIR / 'students.csv', index=False)
print(f'[✓] students.csv — {len(students)} rows')
students.head()

## 3. Generate attendance.csv

In [ ]:
MONTH_LABELS = {1: 'Pre-IT1', 2: 'IT1-to-IT2', 3: 'Pre-Final', 4: 'Final Period'}
attendance_rows = []
week_attendance = np.zeros((N, 12))

for i, sid in enumerate(student_ids):
    base_att = latent_discipline[i] * 55 + 30
    if latent_ability[i] > 0.75 and latent_discipline[i] < 0.4:
        base_att -= np.random.uniform(10, 20)
    if latent_effort[i] > 0.7 and latent_ability[i] < 0.4:
        base_att += np.random.uniform(5, 15)

    for month in range(1, 5):
        life_event = np.random.uniform(-15, 0) if np.random.rand() < 0.12 else 0
        dip   = np.random.uniform(10, 25) if month == 2 and latent_discipline[i] < 0.35 else 0
        boost = np.random.uniform(4, 14)  if month == 4 and latent_effort[i] > 0.65 else 0
        pct   = np.clip(base_att - dip + boost + life_event + np.random.normal(0, 9), 0, 100)

        attendance_rows.append({
            'student_id': sid, 'month': month,
            'month_label': MONTH_LABELS[month], 'attendance_pct': round(float(pct), 1),
        })
        week_start = (month - 1) * 3
        for w in range(3):
            week_attendance[i, week_start + w] = np.clip(pct + np.random.normal(0, 4), 0, 100)

attendance = pd.DataFrame(attendance_rows)
attendance.to_csv(DATA_DIR / 'attendance.csv', index=False)
print(f'[✓] attendance.csv — {len(attendance)} rows')
attendance.head()

## 4. Generate scores.csv

In [ ]:
DIFFICULTY = {'Mathematics': -10, 'Physics': -6, 'Programming': 2, 'Data Structures': -4, 'Statistics': -2}
score_rows = []

for i, sid in enumerate(student_ids):
    arch = ARCHETYPES[i]; ab = latent_ability[i]; eff = latent_effort[i]
    mom = latent_momentum[i]; att_w = week_attendance[i]

    for subj in SUBJECTS:
        diff = DIFFICULTY[subj]
        subj_affinity    = np.random.normal(0, 8)
        subj_consistency = np.random.uniform(0.5, 1.5)
        base = ab * 50 + eff * 22 + 28 + diff + subj_affinity

        weekly = []
        for w in range(12):
            att_effect = (att_w[w] - 65) * 0.12
            if   arch == 0: traj = mom * w * 0.25 + np.random.normal(0, 2)
            elif arch == 1: traj = -8 + w * 1.4 + np.random.normal(0, 3)
            elif arch == 2: traj = 6 - w * 0.9 + max(0, (w-9)*1.2) + np.random.normal(0, 3)
            elif arch == 3: traj = -10 - w * 0.3 + np.random.normal(0, 5)
            elif arch == 4:
                if w < 3: traj = np.random.normal(0, 3)
                elif w < 8: traj = np.random.uniform(-18, -8)
                else: traj = np.random.uniform(5, 14)
            elif arch == 5: traj = -4 + np.random.normal(0, 5)
            else:           traj = 8 - w * 0.3 + np.random.normal(0, 3)
            weekly.append(float(np.clip((base + traj + att_effect) * subj_consistency + np.random.normal(0, 8), 0, 100)))

        anxiety_penalty = latent_anxiety[i] * 10
        it1   = np.clip(weekly[3]  + np.random.normal(-3 - anxiety_penalty, 6), 0, 100)
        it2   = np.clip(weekly[7]  + np.random.normal(-3 - anxiety_penalty, 6), 0, 100)
        final = np.clip(weekly[11] + np.random.normal(-5 - anxiety_penalty, 7), 0, 100)

        if arch == 5:
            p = np.random.uniform(14, 24); b = np.random.uniform(16, 28)
            it1 = np.clip(it1 - p, 0, 100); it2 = np.clip(it2 - p*0.85, 0, 100); final = np.clip(final + b, 0, 100)
        elif arch == 6:
            b = np.random.uniform(8, 16); c = np.random.uniform(22, 38)
            it1 = np.clip(it1 + b, 0, 100); it2 = np.clip(it2 + b*0.75, 0, 100); final = np.clip(final - c, 0, 100)

        slope    = float(np.polyfit([4,8,12], [it1,it2,final], 1)[0])
        weighted = it1 * 0.25 + it2 * 0.25 + final * 0.50
        score_rows.append({
            'student_id': sid, 'subject': subj,
            'it1_score': round(it1,1), 'it2_score': round(it2,1), 'final_score': round(final,1),
            'roll_avg_w4': round(float(np.mean(weekly[0:4])),1),
            'roll_avg_w8': round(float(np.mean(weekly[4:8])),1),
            'roll_avg_w12': round(float(np.mean(weekly[8:12])),1),
            'assignment_avg': round(float(np.mean(weekly)),1),
            'score_trend': round(slope,3), 'weighted_score': round(weighted,1),
        })

scores = pd.DataFrame(score_rows)
scores.to_csv(DATA_DIR / 'scores.csv', index=False)
print(f'[✓] scores.csv — {len(scores)} rows')
scores.head()

## 5. Generate activity.csv

In [ ]:
activity = pd.DataFrame({
    'student_id':          student_ids,
    'lms_logins_per_week': np.clip(latent_discipline*8 + latent_effort*5 + np.random.normal(0,2.5,N), 0, 20).round(1),
    'forum_posts':         np.clip(latent_social*30 + latent_effort*8 + np.random.normal(0,5,N), 0, 60).astype(int),
    'resources_accessed':  np.clip(latent_effort*55 + latent_ability*20 + np.random.normal(0,14,N), 0, 120).astype(int),
    'avg_session_minutes': np.clip(latent_effort*60 + np.random.normal(30,18,N), 5, 180).round(1),
})
activity.to_csv(DATA_DIR / 'activity.csv', index=False)
print(f'[✓] activity.csv — {len(activity)} rows')
activity.head()

In [ ]:
print('Dataset Summary')
print(f'  Students : {len(students)}')
print(f'  Attendance rows : {len(attendance)}')
print(f'  Score rows : {len(scores)}')
print(f'  Activity rows : {len(activity)}')
print('\nAll CSVs saved to', DATA_DIR)